In [ ]:
from pathlib import Path
from itertools import chain, repeat
import re

from natsort import natsorted
import numpy as np
import pandas as pd
from skimage.measure import regionprops_table
from skimage.io import imread
from nd2 import ND2File

def get_num_positions_nd2(in_file):
    """
    get the number of xy-positions / tiles in an nd2 file, will return 1 if file is just a single (potentially multichannel) stack
    """
    with ND2File(in_file) as reader:
        return reader.sizes['P'] if 'P' in reader.sizes else 1

In [ ]:
in_path = '/Volumes/agl_data/AndreasMaiser/NSD/26AM06-02_2'

segmentation_subdirectory = 'segmentation_nuclei'
segmenatation_file_pattern = '[!.]*.tif'

images_subdirectory = 'tif_maxprojection'
image_file_pattern = '[!.]*ch0*.tif'

out_subdirectory = 'region_properties_nuclei'

properties_to_include = ('label', 'area', 'bbox', 'intensity_mean')

# image indexes to extract from filenames and add to table
filename_idxs_to_extract = ('ch', 'pos')

In [ ]:
mask_files = natsorted((Path(in_path) / segmentation_subdirectory).glob(segmenatation_file_pattern))
image_files = natsorted((Path(in_path) / images_subdirectory).glob(image_file_pattern))

# TODO: handle different shape of image / mask (select channel, position, etc..)
# image_files_with_position = list(chain(*(zip(repeat(image_file), range(get_num_positions_nd2(image_file))) for image_file in image_files)))

# mask_files, image_files

In [ ]:
outdir = Path(in_path) / out_subdirectory

if not outdir.exists():
    outdir.mkdir()

for mask_file, image_file in zip(mask_files, image_files):

    # load data and get regionprops
    mask = imread(mask_file)
    img = imread(image_file)
    df = pd.DataFrame(regionprops_table(mask, img, properties=properties_to_include))

    # remember image & mask file (relative to dataset folder)
    df['image_file'] = Path(image_file).relative_to(in_path)
    df['mask_file'] = Path(mask_file).relative_to(in_path)

    # find image, mask idxs in filename (e.g. channel, position) and add to table
    for filename_idx in filename_idxs_to_extract:
        m_image = re.search(f'_{filename_idx}([0-9]+)', image_file.stem)
        m_mask = re.search(f'_{filename_idx}([0-9]+)', mask_file.stem)
        idx_image = int(m_image.groups()[0]) if m_image is not None else np.nan
        idx_mask = int(m_mask.groups()[0]) if m_mask is not None else np.nan
        df[f'image_{filename_idx}'] = idx_image
        df[f'mask_{filename_idx}'] = idx_mask

    outfile = outdir / (image_file.stem + '_regionprops.csv')
    df.to_csv(outfile, index=None)

### Testing stuff

In [ ]:
# Path(image_files[0]).relative_to(in_path)

# mask_files[0].name

# re.match('.*?(_ch[0-9]+_)*_?(_pos[0-9]+_)*.*?', mask_files[0].name).groups()

# m_mask = re.search('_ch([0-9]+)', mask_files[0].name) 
# int(m_mask.groups()[0])